In [1]:
from pyscf import gto, fci, mcscf
import numpy as np
from gpu4pyscf import scf


/home/mert/.local/lib/python3.10/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


In [2]:
# Molecular Information part
# !!!!! in CI part mo not forget the change electrons 
# !!!!! also orbitals are important  

xyz = '''
C   -1.197053   -0.347902   -0.000000
C   -0.785971    0.929486   -0.000000
C    0.785631    0.929728   -0.000000
C    1.197162   -0.347554    0.000000
C    0.000224   -1.156864   -0.000000
H    0.000390   -2.242540    0.000000
H   -2.204763   -0.727422   -0.000000
H   -1.386564    1.827434   -0.000000
H    1.385963    1.827856    0.000000
H    2.205015   -0.726692    0.000000'''
molecule = 'C5H5_cat'
spin = 0
charge =  1


In [4]:
def rhf_calculation(xyz, molecule, spin, charge):
    mol = gto.M(
          atom= xyz,
          basis = 'sto-6g',
          charge = charge,
          verbose = 5 ,
          spin = spin, 
          symmetry = True,
          output  ="./"+molecule+".txt")

    mf = mol.RHF().run()
    mf.analyze() 
    S_ij = mf.get_ovlp()
    C_ij = mf.mo_coeff
    np.savetxt("C_"+molecule+'mo_coeff.txt', C_ij,fmt = '%.8f')
    np.savetxt('S_'+molecule+'_overlap.txt', S_ij,fmt = '%.8f')

    return mf, mol 

In [5]:
def orb_occ_table(orb_energy,orb_occ):
    Homo = None
    Lumo = None
    print ("orbital  energy   occ")

    for i in range (0,(len(orb_occ))):
        print(i ,"   ", orb_energy[i],"  ",orb_occ[i])
              
    for i in range(len(orb_occ) - 1):
        if orb_occ[i] == 2  and orb_occ[i+1] == 0:
            homo = i
            lumo = i + 1
            print(f"Homo_index: {homo}, Lumo_index: {lumo}")
    return homo, lumo


In [6]:
mf, mol  = rhf_calculation( xyz , molecule, spin, charge )
homo , lumo =orb_occ_table(mf.mo_energy, mf.mo_occ)

output file: ./C5H5_cat.txt
orbital  energy   occ
0     -11.58626912884549    2.0
1     -11.507113729796599    2.0
2     -11.50704411735201    2.0
3     -11.438719600589156    2.0
4     -11.438673210051041    2.0
5     -1.3939451447514974    2.0
6     -1.1994436635069725    2.0
7     -1.1623385363566134    2.0
8     -0.9641358832590828    2.0
9     -0.9634135417028352    2.0
10     -0.9229625122541648    2.0
11     -0.791338997587777    2.0
12     -0.772650729832825    2.0
13     -0.7533208819251404    2.0
14     -0.7255759656155537    2.0
15     -0.6942554175544275    2.0
16     -0.535288023739087    2.0
17     -0.19395778439070208    0.0
18     0.1058935088014204    0.0
19     0.11382121619617903    0.0
20     0.26732646683850403    0.0
21     0.3342520254775443    0.0
22     0.38515718428974693    0.0
23     0.4099536493381115    0.0
24     0.44284529150555707    0.0
25     0.5006646264854122    0.0
26     0.5417966743256495    0.0
27     0.6289058687484288    0.0
28     0.782663056

WARN: System has nonzero charge 1; the dipole moment is origin-dependent.
Location of origin: [0. 0. 0.]


In [7]:
n_elec= 4
n_orb = 5 # number of orbital can be determined based on the number of pi  orbitals 
casscf = mcscf.CASSCF(mf, n_orb, n_elec)
'''output in order
e_tot
e_cas
Fci vecs 
mo_coeff
mo_energy'''
mo = casscf.sort_mo([16,17,18,19,20])# the numbers are from the index of pi orbitals 
casscf.kernel(mo) 


(np.float64(-191.54260516911603),
 np.float64(-4.452771181738768),
 FCIvector([[-9.35328595e-01,  2.43601922e-08, -2.86944291e-02,
             -4.44785982e-02,  3.01737934e-05, -2.42562306e-02,
             -7.78199731e-06,  2.96573030e-02,  2.30923894e-05,
             -1.13789478e-03],
            [ 2.43601922e-08,  1.21045431e-01, -3.12557778e-05,
              5.27723358e-06,  7.34012708e-02, -9.64546309e-06,
             -7.62041610e-02, -5.23649565e-05, -7.04439042e-03,
             -2.36122654e-06],
            [-2.86944291e-02, -3.12557778e-05,  1.32474811e-01,
              9.76572591e-02, -8.70266905e-06, -4.07922076e-03,
             -7.54927675e-05, -1.45042478e-02,  5.92178254e-06,
             -6.22304571e-04],
            [-4.44785982e-02,  5.27723358e-06,  9.76572591e-02,
              1.03812800e-01, -3.64495170e-05, -3.26928216e-03,
             -2.56351504e-05, -6.23132011e-02,  1.02839108e-06,
             -1.42715444e-03],
            [ 3.01737934e-05,  7.34012708

In [8]:
casscf.fcisolver.max_cycle = 3000
casscf.canonicalize(mo_coeff=None, ci=None, eris=None, sort=False, cas_natorb=False, casdm1=None, verbose=3, with_meta_lowdin=False);

filename = f"{molecule}_Ci_coeffs.txt"

with open(filename, 'w', encoding='utf-8') as f:
    
    f.write("det-alpha,     det-beta,    CI coefficients\n")
    
    
    for c, ia, ib in casscf.fcisolver.large_ci(casscf.ci, n_orb, n_elec, tol=.000000001, return_strs=False):
        # Satır formatını oluşturalım
        line = f" {str(ia):<12} {str(ib):<12} {c:.15f}\n"
        
        #write in file 
        f.write(line)
        #show in terminal 
        print(line.strip())

[0 1]        [0 1]        -0.935328594872940
[0 1]        [0 2]        0.000000024360192
[0 1]        [1 2]        -0.028694429092903
[0 1]        [0 3]        -0.044478598194803
[0 1]        [1 3]        0.000030173793420
[0 1]        [2 3]        -0.024256230627061
[0 1]        [0 4]        -0.000007781997308
[0 1]        [1 4]        0.029657303044753
[0 1]        [2 4]        0.000023092389377
[0 1]        [3 4]        -0.001137894775690
[0 2]        [0 1]        0.000000024360192
[0 2]        [0 2]        0.121045431279102
[0 2]        [1 2]        -0.000031255777788
[0 2]        [0 3]        0.000005277233580
[0 2]        [1 3]        0.073401270797146
[0 2]        [2 3]        -0.000009645463093
[0 2]        [0 4]        -0.076204161033127
[0 2]        [1 4]        -0.000052364956540
[0 2]        [2 4]        -0.007044390420654
[0 2]        [3 4]        -0.000002361226536
[1 2]        [0 1]        -0.028694429092903
[1 2]        [0 2]        -0.000031255777788
[1 2]        [1 2]